# Circularly-permuted construct: two rigid bodies + mixed distance constraints

This notebook builds a Carbonara run for a circularly-permuted sensor construct
(sensing domain N-term → linker → GFP → linker → sensing domain C-term), where:

- The two linkers are cut at fixed residues (only the literal cut, not the whole loop —
  the rest of each loop stays in the model so it still contributes to the SAXS curve).
  The removed stretch is meant to be rebuilt later once new structures are generated.
- Sensing-N (1–313) and Sensing-C (563–620) are joined into **one rigid body**
  (this reconstructs the natural, uninterrupted sensing domain — that's what circular
  permutation means: what were the real N- and C-termini of the sensing domain are now
  the two cut ends either side of the GFP insertion).
- GFP (316–559) is the **second rigid body**.
- Both rigid bodies have **zero internal flexibility** — `--rotation` is the only way
  either one can move, so the search is purely over their relative
  position/orientation, not internal reshaping.
- The junction at 313↔316 (where linker 1 was) gets a **strict** distance constraint,
  since removing that gap must not let the model drift apart there.
- The junction at 560↔563 (where linker 2 was) gets **no constraint** — you noted
  there's a lot of wiggle room there physically, so it's left fully free.
- Six DSSO crosslink pairs get **soft, upper-bound-only** constraints: violated only if
  the modelled distance exceeds the derived bound, never if it's closer (a crosslink is a
  reachability bound, not a target).

The upper-bound-only constraint type is new — added to `ktlMoleculeRandom.cpp` this
session (`loadContactPredictions`/`getLennardJonesContact`) as an optional 7th column on
`fixedDistanceConstraints1.dat` (0 = old symmetric-target behaviour, 1 = upper-bound-only).
It's backward compatible: existing 6-column constraint files still parse and behave exactly
as before. The Carbonara binaries have already been rebuilt with this patch.

All of the residue numbers, targets and tolerances below are variables — check them
against your actual construct before running, especially the exact linker cut points
(see the note in Step 1).

## Step 0 — Configure

`sensing_n`, `sensing_c`, `gfp`: the three fragments to keep, as
`(start_resid, end_resid)` in the **original** PDB numbering. Everything not covered by
one of these three ranges (i.e. the two linkers) is dropped from the model.

The residue ranges below implement your literal cut: 314–315 removed (linker 1, strict
post at 313↔316) and 560–562 removed (linker 2, unconstrained). If your construct's real
junction residues differ, change these four numbers — nothing else in this cell depends
on the specific values.

In [2]:
import CarbonaraDataTools as CDT
import numpy as np
import pickle
import sys
from pathlib import Path

run_name  = "circular_permutant_MBP_TEST01"
pdb_name  = "pdbFiles/MBP_apo_Bilbo_2.pdb"      # put your structure here
saxs_name = "saxsFiles/MBP_2_qp3.dat"  # put your SAXS data here

# Fragment residue ranges, original PDB numbering, inclusive.
sensing_n = (1,   313)   # sensing domain, N-terminal side of the permutation
sensing_c = (563, 620)   # sensing domain, C-terminal side of the permutation
gfp       = (316, 559)   # GFP

# Write order matters: sensing_n then sensing_c then gfp, so that after setup the
# two sensing-domain fragments are chains 1 and 2 (physically ADJACENT in
# coordinates1.dat) and can be safely merged with merge_pair=(1,2) in Step 2.
# Carbonara's own chain-merge helpers relabel chain *boundaries* in fingerPrint1.dat
# but never touch coordinates1.dat -- merging non-adjacent chains would silently
# misalign coordinates against the new chain boundaries. Keeping the two
# to-be-merged fragments adjacent in the write order sidesteps that entirely.
fragments = [("A", *sensing_n), ("B", *sensing_c), ("C", *gfp)]

construct_pdb = f"pdbFiles/{run_name}_construct.pdb"

## Step 1 — Build the 3-fragment construct PDB

Reassigns chain IDs by residue range and drops everything outside the three fragments
(i.e. the two linker windows). Plain text PDB parsing — no dependency on DSSP/mdtraj at
this stage, so you can see exactly which atoms go where.

**Check before running for real:** the section below prints the secondary-structure
context around each cut point (via Carbonara's own DSSP-based segmenter) purely so you
can confirm 313/316 and 560/563 don't fall mid-helix or mid-strand for *your* structure.
It does not change the cut — the literal residues from Step 0 are always what's used.

In [3]:
def build_fragment_pdb(in_pdb, out_pdb, fragments):
    """fragments: list of (chain_id, start_resid, end_resid), written in this order."""
    lines_by_frag = {i: [] for i in range(len(fragments))}
    with open(in_pdb) as f:
        atom_lines = [l for l in f if l.startswith(("ATOM", "HETATM"))]
    for l in atom_lines:
        resid = int(l[22:26])
        for i, (cid, start, end) in enumerate(fragments):
            if start <= resid <= end:
                lines_by_frag[i].append(l[:21] + cid + l[22:])
                break
    with open(out_pdb, "w") as f:
        for i in range(len(fragments)):
            f.writelines(lines_by_frag[i])
            f.write("TER\n")
        f.write("END\n")
    return {cid: (end - start + 1) for cid, start, end in fragments}

frag_lengths = build_fragment_pdb(pdb_name, construct_pdb, fragments)
print("fragment residue counts:", frag_lengths)

fragment residue counts: {'A': 313, 'B': 58, 'C': 244}


In [4]:
# Optional sanity check: confirm the cut points sit in loop/coil regions, not
# mid-helix or mid-strand, using Carbonara's own DSSP-based section detector.
sys.path.insert(0, "../deploy/api")  # adjust if this repo layout differs
import sections as _sections

r = _sections.analyse(Path(pdb_name))
for s in r["sections"]:
    if s["start"] is None:
        continue
    near_break1 = s["end"] >= sensing_n[1] - 15 and s["start"] <= gfp[0] + 15
    near_break2 = s["end"] >= gfp[1] - 15 and s["start"] <= sensing_c[0] + 15
    if near_break1 or near_break2:
        print(s)

{'index': 37, 'type': '-', 'chain': 0, 'start': 298, 'end': 316, 'length': 19, 'selectable': True, 'reason': None, 'terminal': False}
{'index': 38, 'type': 'S', 'chain': 0, 'start': 317, 'end': 322, 'length': 6, 'selectable': False, 'reason': 'sheet', 'terminal': False}
{'index': 39, 'type': '-', 'chain': 0, 'start': 323, 'end': 328, 'length': 6, 'selectable': True, 'reason': None, 'terminal': False}
{'index': 40, 'type': 'S', 'chain': 0, 'start': 329, 'end': 339, 'length': 11, 'selectable': False, 'reason': 'sheet', 'terminal': False}
{'index': 67, 'type': '-', 'chain': 0, 'start': 542, 'end': 565, 'length': 24, 'selectable': True, 'reason': None, 'terminal': False}
{'index': 68, 'type': 'H', 'chain': 0, 'start': 566, 'end': 576, 'length': 11, 'selectable': False, 'reason': 'helix', 'terminal': False}
{'index': 69, 'type': '-', 'chain': 0, 'start': 577, 'end': 585, 'length': 9, 'selectable': True, 'reason': None, 'terminal': False}


## Step 2 — Run setup with `--rotation`

Standard Carbonara setup, on the 3-fragment construct, with whole-chain rigid rotation
enabled. This produces the usual `fingerPrint1.dat` / `coordinates1.dat` /
`chainLengths.dat` / `varyingSectionSecondary1.dat` / `RunMe_<run_name>.sh` under
`carbonara_runs/<run_name>/`.

At this point there are still **3** Carbonara chains (sensing-N, sensing-C, GFP) —
they get merged down to 2 in Step 3.

In [8]:
# !{sys.executable} setup_carbonara_allAtom.py -p $construct_pdb -s $saxs_name -n $run_name --rotation --distance_constraint_cap 50.0

!{sys.executable} setup_carbonara_allAtom.py -p $construct_pdb -s $saxs_name -n $run_name --rotation --distance_constraint_cap 50.0 --max_q 0.2 --max_q_start 0.2 --fit_n_times 4

run_dir = f"carbonara_runs/{run_name}"

Created directory structure in: /Users/josh/Downloads/carbonara-web/carbonara/carbonara_runs/circular_permutant_MBP_TEST01
The number of chains is  3
Are sure you have more than one chain - if not this will cause segmentation errors later! You have been warned...
100%|███████████████████████████████████████████| 38/38 [00:07<00:00,  4.96it/s]

Setup completed successfully!
Initial files were created in: /Users/josh/Downloads/carbonara-web/carbonara/carbonara_runs/circular_permutant_MBP_TEST01
Files for Carbonara were copied to: /Users/josh/Downloads/carbonara-web/carbonara/carbonara_runs/circular_permutant_MBP_TEST01
Run script created at: /Users/josh/Downloads/carbonara-web/carbonara/RunMe_circular_permutant_MBP_TEST01.sh

To run the refinement, execute:
cd /Users/josh/Downloads/carbonara-web/carbonara && ./RunMe_circular_permutant_MBP_TEST01.sh


## Step 3 — Zero out internal flexibility, then merge the sensing-domain fragments

Two edits to the freshly-written model files:

1. **Clear `varyingSectionSecondary1.dat` completely.** With no candidate loops selected
   on *either* chain, nothing can be internally reshaped — the only motion left is the
   `--rotation` rigid-body transform, which is exactly "two rigid bodies, no resampling."
2. **Merge chains 1 and 2** (sensing-N and sensing-C) into a single rigid chain, using the
   same helpers the IgG2 multimer notebook uses
   (`create_segment_label_arrays_with_merge_v4` / `merge_chains_only_clean_consistent_segments`).
   Since we already cleared the varying-section list, there's nothing left to remap —
   this step purely reshapes the chain/segment bookkeeping so `--rotation` treats
   sensing-N + sensing-C as one unit and GFP as the other.

After this, `chainLengths.dat` still describes the **pre-merge** 3-chain layout (A, B, C)
— it isn't regenerated by the merge. That's fine and deliberate: because sensing-N and
sensing-C were written adjacently in Step 1, merging them doesn't shift any residue's
position in the flat coordinate file, so the pre-merge, per-chain offsets in
`chainLengths.dat` still give the right global positions in Step 4. This only holds
because the merged pair is physically adjacent — it would not be safe for a
non-adjacent merge.

In [9]:
# 1. clear all varying sections
open(f"{run_dir}/varyingSectionSecondary1.dat", "w").close()

# 2. merge chain 1 (sensing-N) + chain 2 (sensing-C)
chains = CDT.parse_structures_with_segments(f"{run_dir}/fingerPrint1.dat")
merge_pair = (1, 2)
_, _, filtered = CDT.create_segment_label_arrays_with_merge_v4(
    chains, highlighted_segments=np.array([], dtype=int), merge_pair=merge_pair
)
merged_chains = CDT.merge_chains_only_clean_consistent_segments(chains, merge_pair)

CDT.export_chains_to_file(merged_chains, f"{run_dir}/fingerPrint1.dat")
CDT.export_segment_list(set(filtered.tolist()), f"{run_dir}/varyingSectionSecondary1.dat")

print("chains after merge:", [len(c["sequence"]) for c in merged_chains])
# expect: [len(sensing_n)+len(sensing_c), len(gfp)]

chains after merge: [371, 244]


## Step 4 — Distance constraints

Two groups, written to `fixedDistanceConstraints1.dat` in one call so they don't
overwrite each other:

- **The strict post** (313 ↔ 316, where linker 1 was cut): target = the actual measured
  distance in your starting structure, very small tolerance so it effectively can't drift.
  `bound_type=0` (two-sided) — appropriate here since this is a real physical anchor, not
  a reachability bound.
- **The six XL-MS pairs**: DSSO's ~17.2Å CB-CB reach, converted to a looser CA-CA upper
  bound with a flat +3.0Å buffer (CB sits ~1.5Å off the CA on each residue, worst case
  additive) → 20.2Å. `bound_type=1` (upper-bound-only, the new constraint type) so being
  closer than 20.2Å is never penalised — only exceeding it is. Tolerance is large (soft/
  "encourage" rather than "enforce") — tune `xl_tolerance` to taste; smaller = more
  insistent.

Residue numbers below are **original PDB numbering** — the cell converts them to
Carbonara's internal chain-offset positions for you using `chainLengths.dat`, so you never
have to hand-compute local indices.

In [10]:
# original-PDB-numbering -> (carbonara chain letter, local 1-indexed position)
chain_lengths = pickle.load(open(f"{run_dir}/chainLengths.dat", "rb"))
chain_order = sorted(chain_lengths.keys())  # ['A','B','C'] = sensing_n, sensing_c, gfp
chain_offsets, _off = {}, 0
for ch in chain_order:
    chain_offsets[ch] = _off
    _off += chain_lengths[ch]

def resid_to_global(resid):
    for (cid, start, end), letter in zip(fragments, chain_order):
        if start <= resid <= end:
            local = resid - start + 1
            return chain_offsets[letter] + local
    raise ValueError(f"residue {resid} is not in any kept fragment (was it cut out?)")

# --- the strict post ---
# hard=True: a true feasibility filter now (requires the rebuilt engine from this
# session) -- moves that would violate this beyond post_tolerance are rejected outright,
# no matter how much chi2 improves. That's what actually makes it unbreakable; an
# ultra-small tolerance alone (the old approach) only ever made breaking it *expensive*,
# and a big enough chi2 gain could still buy past it. post_tolerance can now be a normal,
# physically-reasonable value instead of needing to be pushed to extremes.
post_pair = (sensing_n[1], gfp[0])   # (313, 316)
post_tolerance = 0.05                # fraction of target distance -- hard, so this can be sane
post_hard = True

# --- the XL-MS pairs (residue A -> residue B), DSSO ---
# hard=False (soft): bounded/capped penalty (also from this session) -- encourages without
# ever being able to explode regardless of how tight xl_tolerance is set, and can still be
# outweighed by a big enough chi2 gain, matching "encourage, don't kill".
xl_pairs = {65: 359, 62: 365, 61: 365, 155: 369, 51: 549}
cb_cb_reach   = 17.2   # measured/assumed DSSO CB-CB reach, Angstrom
ca_ca_buffer  = 3.0    # CB-offset allowance, Angstrom -- tune if you'd rather predict sidechain COM
xl_bound      = cb_cb_reach + ca_ca_buffer   # 20.2 Angstrom CA-CA upper bound
xl_tolerance  = 0.2     # soft -- encourages, never kills the fit. Tune to taste.
xl_hard = False

coords = np.genfromtxt(f"{run_dir}/coordinates1.dat")

contactPreds, fixedDists, tolerances, boundTypes, hardList = [], [], [], [], []

# post: measure the actual current distance as the target
gA, gB = resid_to_global(post_pair[0]), resid_to_global(post_pair[1])
post_dist = float(np.linalg.norm(coords[gA - 1] - coords[gB - 1]))
contactPreds.append([gA, gB]); fixedDists.append(post_dist)
tolerances.append(post_tolerance); boundTypes.append(0); hardList.append(int(post_hard))

# XL-MS pairs: fixed upper-bound target, not measured from this (unrelated) apo structure
for a, b in xl_pairs.items():
    contactPreds.append([resid_to_global(a), resid_to_global(b)])
    fixedDists.append(xl_bound)
    tolerances.append(xl_tolerance)
    boundTypes.append(1)
    hardList.append(int(xl_hard))

print(f"post: residues {post_pair} -> global {[gA, gB]}, target={post_dist:.2f} A, tol={post_tolerance}, hard={post_hard}")
for (a, b), (g1, g2), d, t, bt in zip(list(xl_pairs.items()), contactPreds[1:], fixedDists[1:], tolerances[1:], boundTypes[1:]):
    print(f"XL: residues ({a},{b}) -> global ({g1},{g2}), bound<={d} A, tol={t}, upper_bound_only={bool(bt)}, hard={xl_hard}")

CDT.translate_distance_constraints(contactPreds, coords, run_dir, fixedDists, tolerances, boundTypes, hardList)
CDT.toggle_paired_predictions(f"RunMe_{run_name}.sh")


post: residues (313, 316) -> global [313, 372], target=11.77 A, tol=0.05, hard=True
XL: residues (65,359) -> global (65,415), bound<=20.2 A, tol=0.2, upper_bound_only=True, hard=False
XL: residues (62,365) -> global (62,421), bound<=20.2 A, tol=0.2, upper_bound_only=True, hard=False
XL: residues (61,365) -> global (61,421), bound<=20.2 A, tol=0.2, upper_bound_only=True, hard=False
XL: residues (155,369) -> global (155,425), bound<=20.2 A, tol=0.2, upper_bound_only=True, hard=False
XL: residues (51,549) -> global (51,605), bound<=20.2 A, tol=0.2, upper_bound_only=True, hard=False


## Step 5 — Run

Same as the other notebooks — starts the search and the live monitor
(all-atom backmapping + FoXS evaluation of generated structures).

In [ ]:
from pathlib import Path
from run_frontend import CarbonaraRunner

foxs_cmd = f'python3 {Path("external/pyFoXS/pyFoXS/foxs.py").resolve()}'

runner = CarbonaraRunner(run_name, foxs_cmd=foxs_cmd)
runner.start()
runner.start_monitor(threshold=2.5, every_s=10, defer_backmap_seconds=600)

ℹ️ No backmapping method detected in run script; skipping MODELLER preflight check
🚀 Carbonara started (PID=2435)


## 🔬 Carbonara Live Monitor

**Status:** Backmapping window active. New eligible predictions should now be picked up.
**Threshold (χ²):** 2.5
**Mixture components:** 1
**Good models:** 0 / 0
**Errors:** 0
**Last sweep:** 18:13:01

📊 Monitoring started (mixture_n=1)


In [ ]:
import numpy as np
import plotly.graph_objs as go

def read_carbonara_xyz(path):
    """Robust to 'End chain N' delimiter lines -- returns only real coordinate rows,
    in file order (row index = global position - 1)."""
    rows = []
    for line in open(path):
        parts = line.split()
        if len(parts) != 3:
            continue
        try:
            rows.append([float(p) for p in parts])
        except ValueError:
            continue
    return np.array(rows)

def kabsch_align(mobile, reference):
    """Best rotation+translation (no reflection, no scaling) superposing mobile onto
    reference. mobile/reference: paired Nx3 arrays -- same atoms, same order."""
    mobile_c = mobile - mobile.mean(axis=0)
    ref_c = reference - reference.mean(axis=0)
    H = mobile_c.T @ ref_c
    U, S, Vt = np.linalg.svd(H)
    d = np.sign(np.linalg.det(Vt.T @ U.T))
    D = np.diag([1.0, 1.0, d])   # flips the improper (reflection) solution back to a true rotation
    R = Vt.T @ D @ U.T
    t = reference.mean(axis=0) - R @ mobile.mean(axis=0)
    return R, t

def apply_transform(coords, R, t):
    return (R @ coords.T).T + t

def align_on_sensing_domain(mobile_full, reference_full, sensing_slice):
    """Superpose mobile_full onto reference_full using only the sensing-domain rows
    to compute the transform, then apply it to every row (sensing + GFP) -- so the
    sensing domains overlap and any remaining GFP displacement is real relative motion."""
    R, t = kabsch_align(mobile_full[sensing_slice], reference_full[sensing_slice])
    return apply_transform(mobile_full, R, t)

# --- sensing-domain / GFP row ranges, from the merge in Step 3 ---
# after merging, chain 1 = sensing-N + sensing-C (rows 0 .. n_sensing-1), chain 2 = GFP
n_sensing = frag_lengths["A"] + frag_lengths["B"]   # sensing_n + sensing_c residue counts
sensing_slice = slice(0, n_sensing)
gfp_slice     = slice(n_sensing, n_sensing + frag_lengths["C"])

# --- load structures ---
pred_path = '/Users/josh/Downloads/carbonara-web/carbonara/carbonara_runs/circular_permutant_MBP_TEST01/fitdata/mol2_sub_0_step_1_xyz.dat'
init_path = '/Users/josh/Downloads/carbonara-web/carbonara/carbonara_runs/circular_permutant_MBP_TEST01/fitdata/mol2_sub_0_initial_xyz.dat'

data_init = read_carbonara_xyz(init_path)
data_pred_raw = read_carbonara_xyz(pred_path)

# use the initial structure's sensing domain as the fixed reference frame
data_pred = align_on_sensing_domain(data_pred_raw, data_init, sensing_slice)

# --- plot: sensing domain vs GFP as separate traces, so alignment quality and
#     barrel movement are both visible at a glance ---
fig = go.Figure()

def add_structure(fig, data, name, sensing_color, gfp_color):
    fig.add_trace(go.Scatter3d(
        x=data[sensing_slice,0], y=data[sensing_slice,1], z=data[sensing_slice,2],
        mode='lines', line=dict(width=5, color=sensing_color), name=f'{name} sensing domain'
    ))
    fig.add_trace(go.Scatter3d(
        x=data[gfp_slice,0], y=data[gfp_slice,1], z=data[gfp_slice,2],
        mode='lines', line=dict(width=4, color=gfp_color), name=f'{name} GFP'
    ))

add_structure(fig, data_init, 'initial',   'darkred',  'red')
add_structure(fig, data_pred, 'predicted', 'darkblue', 'dodgerblue')

fig.update_layout(
    title='Sensing-domain-aligned: GFP barrel movement',
    scene=dict(xaxis_title='X', yaxis_title='Y', zaxis_title='Z', aspectmode='data'),
    margin=dict(l=0, r=0, b=0, t=40)
)
fig.show()

In [ ]:
def crosslink_distance_report(data, pairs, resid_to_global=resid_to_global):
    """pairs: list of (label, res_a, res_b, bound_type, target_or_bound, tolerance).
    Returns a list of dicts with the measured distance and whether the constraint
    is satisfied, and prints a formatted table."""
    rows = []
    print(f"{'label':<14}{'residues':<12}{'distance':>10}{'target/bound':>14}{'satisfied':>12}")
    for label, res_a, res_b, bound_type, target, tol in pairs:
        pa = data[resid_to_global(res_a) - 1]
        pb = data[resid_to_global(res_b) - 1]
        dist = float(np.linalg.norm(pa - pb))
        if bound_type == 1:  # upper-bound-only: satisfied if within the bound
            ok = dist <= target
        else:                # two-sided target: satisfied if close, using tolerance*target as a rough band
            ok = abs(dist - target) <= tol * target
        rows.append({"label": label, "res_a": res_a, "res_b": res_b, "distance": dist,
                     "target": target, "bound_type": bound_type, "satisfied": ok})
        print(f"{label:<14}{f'{res_a}-{res_b}':<12}{dist:>9.2f}Å{target:>12.2f}Å{str(ok):>12}")
    return rows

# reuse the exact constraint definitions from Step 4 (post_pair, post_dist, post_tolerance,
# xl_pairs, xl_bound, xl_tolerance) -- run that cell first if these aren't in scope
labeled_pairs = [("post", *post_pair, 0, post_dist, post_tolerance)] + \
                [(f"XL {a}-{b}", a, b, 1, xl_bound, xl_tolerance) for a, b in xl_pairs.items()]

print("=== initial structure ===")
report_init = crosslink_distance_report(data_init, labeled_pairs)

print("\n=== predicted structure (sensing-domain aligned) ===")
report_pred = crosslink_distance_report(data_pred, labeled_pairs)

=== initial structure ===
label         residues      distance  target/bound   satisfied
post          313-316         11.77Å       11.77Å        True
XL 65-359     65-359          58.46Å       20.20Å       False
XL 62-365     62-365          39.31Å       20.20Å       False
XL 61-365     61-365          41.41Å       20.20Å       False
XL 155-369    155-369         45.87Å       20.20Å       False
XL 51-549     51-549          52.61Å       20.20Å       False

=== predicted structure (sensing-domain aligned) ===
label         residues      distance  target/bound   satisfied
post          313-316         12.18Å       11.77Å        True
XL 65-359     65-359          49.53Å       20.20Å       False
XL 62-365     62-365          33.34Å       20.20Å       False
XL 61-365     61-365          36.16Å       20.20Å       False
XL 155-369    155-369         40.30Å       20.20Å       False
XL 51-549     51-549          51.75Å       20.20Å       False


In [65]:
# Run this cell to stop.
runner.stop()
runner.stop_monitor()

🛑 Stopping Carbonara...
✅ Stopped
📊 Monitoring stopped


## Notes / things worth re-checking before a real run

- **Exact cut residues**: the DSSP check in Step 1 tells you whether 313/316 and 560/563
  land in loop regions of *this* starting structure — it doesn't know where your genetic
  construct's real linker boundaries are. If your fusion junction is a residue or two off
  from these numbers, just change `sensing_n` / `sensing_c` / `gfp` in Step 0.
- **`xl_bound` / `ca_ca_buffer`**: the 20.2Å figure is a simple flat conversion from the
  DSSO CB-CB reach, not a per-pair sidechain-geometry prediction. If some of the six pairs
  have unusual sidechain orientations you know about, you can pass a per-pair list instead
  of a single constant — `fixedDists`/`tolerances`/`boundTypes` in Step 4 are already
  built as plain per-pair Python lists, so this is a one-line change.
- **`xl_tolerance`**: this is the main knob for how "soft" the crosslinks are. Since the
  penalty scales as `((actual-target)/target/tolerance)^4`, doubling the tolerance makes a
  given violation contribute ~16x less — it's a steep curve, so small changes here have a
  large effect on how insistent the restraint feels relative to chi2.
- **Rebuilding the cut linkers**: this notebook deliberately leaves 314–315 and 560–562
  out of the model entirely (not just unconstrained — physically absent), per your note
  that you'll rebuild them once new structures are generated. That means the SAXS curve
  predicted during the search is missing those ~5 residues of scattering mass — fine for
  the search itself, but worth remembering when comparing predicted vs experimental I(q)
  before the linkers are rebuilt back in.

## Step 6 — Loss dashboard: what's actually improving

The raw `fitLog*.dat` JSON is hard to scan by eye, and (before this session) `ScatterFitFirst`
wasn't even real chi2 -- it's the fully combined, penalty-weighted objective. This reads a
fitLog and renders the last N accepted moves as a table with per-column deltas, so you can see
at a glance which terms are trending the right way and whether the hard post is under any
strain.

In [24]:
import json

def read_fitlog(path):
    entries = []
    for line in open(path):
        line = line.strip()
        if not line.startswith("{") or '"ImprovementIndex"' not in line:
            continue
        try:
            entries.append(json.loads(line))
        except Exception:
            continue
    return entries

def show_fitlog_dashboard(path, last_n=15):
    entries = read_fitlog(path)
    if not entries:
        print("no entries yet")
        return
    entries = entries[-last_n:]
    cols = ["ImprovementIndex", "FitStep", "Chi2", "DistanceConstraints", "HardConstraintsSatisfied", "HardConstraintsMaxViolation"]
    header = f"{'ImpvIdx':>8} {'Step':>6} {'Chi2':>12} {'ChiΔ':>10} {'SoftDist':>10} {'DistΔ':>10} {'Hard':>10}"
    print(header)
    print("-" * len(header))
    prev = None
    for e in entries:
        chi2 = e.get("Chi2", float("nan"))
        dist = e.get("DistanceConstraints", 0.0)
        chi2_d = "" if prev is None else f"{chi2 - prev.get('Chi2', float('nan')):+.4f}"
        dist_d = "" if prev is None else f"{dist - prev.get('DistanceConstraints', 0.0):+.4f}"
        hard = "OK" if e.get("HardConstraintsSatisfied", True) else f"VIOL({e.get('HardConstraintsMaxViolation', 0):.3f})"
        print(f"{e['ImprovementIndex']:>8} {e['FitStep']:>6} {chi2:>12.4f} {chi2_d:>10} {dist:>10.4f} {dist_d:>10} {hard:>10}")
        prev = e

# point this at whichever run you want to inspect
show_fitlog_dashboard(f"{run_dir}/fitdata/fitLog1.dat", last_n=15)


 ImpvIdx   Step         Chi2       ChiΔ   SoftDist      DistΔ       Hard
------------------------------------------------------------------------
       0      0       3.9589                1.4988                    OK
       1     66       2.2396    -1.7193     0.9973    -0.5015         OK
       2    189       0.5440    -1.6956     1.9054    +0.9081         OK
       3    202       0.1304    -0.4135     1.5963    -0.3090         OK


## Step 7 — Pareto front: best chi2/distance trade-offs across the whole run

Rather than trusting whichever step the single-scalar greedy search happened to land on last,
scan every saved trajectory structure and report the **Pareto front** -- structures where no
other saved structure has both better chi2 *and* better soft-distance penalty. This is the
standard approach once you have two genuinely competing objectives: it shows you the actual
best joint trade-offs the search already generated, instead of picking one number to optimize.

Uses `read_carbonara_xyz`/`resid_to_global` from Step 4/6 above -- no new run needed, this
works on structures you already have.

In [1]:
import glob

def pareto_front_for_run(run_directory, xl_pairs_for_scoring=None):
    """Scans every fitLog*.dat in run_directory/fitdata, returns the Pareto-optimal
    (chi2, soft-distance-penalty) entries -- no other entry beats one on both axes."""
    candidates = []
    for log_path in glob.glob(f"{run_directory}/fitdata/fitLog*.dat"):
        for e in read_fitlog(log_path):
            candidates.append({
                "chi2": e.get("Chi2", float("nan")),
                "dist": e.get("DistanceConstraints", 0.0),
                "hard_ok": e.get("HardConstraintsSatisfied", True),
                "path": e.get("MoleculePath"),
                "log": log_path,
                "step": e.get("FitStep"),
            })

    # only consider structures where the hard constraint actually held
    candidates = [c for c in candidates if c["hard_ok"]]

    front = []
    for c in candidates:
        dominated = any(
            (o["chi2"] <= c["chi2"] and o["dist"] <= c["dist"]) and
            (o["chi2"] < c["chi2"] or o["dist"] < c["dist"])
            for o in candidates
        )
        if not dominated:
            front.append(c)

    front.sort(key=lambda c: c["chi2"])
    return front

front = pareto_front_for_run(run_dir)
print(f"{len(front)} Pareto-optimal structures (of the hard-constraint-satisfying candidates):")
for c in front:
    print(f"  chi2={c['chi2']:.4f}  soft_dist_penalty={c['dist']:.4f}  {c['path']}")


NameError: name 'run_dir' is not defined